# Species ETL: manual mapping → SEAD taxon/common-name matches

`output/species_split_taxa_gbif_matches.csv` is the automated match of raw `species_split` tokens
against SEAD taxonomy + GBIF built in `species_study.ipynb`. Many rows there are unmatched or
ambiguous because the raw tokens are misspellings, duplicates, or compound Swedish names the
heuristics couldn't resolve.

`data/species_split_counts_in_original_manual.csv` is a hand-built correction of that vocabulary:
every `species_split` token mapped to a corrected `manual_species` value (typos fixed, duplicates
merged, uncertain calls marked with a trailing `?`, a few tokens split into multiple candidate
species via commas). This notebook takes that corrected, smaller vocabulary through its own fresh
match against SEAD + GBIF, rather than inheriting the old per-token results as-is:

1. Re-derive the vocabulary + counts (comma-split, recount).
2. Carry over whatever GBIF/SEAD match info already exists for it.
3. Try a direct SEAD match (common name, then Latin genus/family/order tables) on the corrected
   spelling before touching the network.
4. Fill remaining gaps with fresh GBIF lookups.
5. Resolve/propose SEAD `taxon_id`/`common_name_id` for every row that now has a taxonomic
   anchor, flagging the rest for manual review.

The `sead_staging` connection used here is read-only, so step 5's output is a *proposal* — new
IDs computed as `max(existing_id)+1` — for later manual ingestion, not live INSERTs.

In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

OUTPUT_DIR = Path('../output/species')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Re-derive the manual vocabulary + counts

Load the manual mapping, split any comma-separated `manual_species` into one row per candidate
(each getting the *full* original count — not divided — same convention `species_study.ipynb`
used when an original `species` value listed multiple candidates), keep `?`-marked values distinct
from their unmarked counterpart, and keep the blank row (no `species_split`/`manual_species` at
all) rather than dropping it — it's the count of rows with no species value, same role as the
`<NA>` row in the old notebook's counts.

In [2]:
manual_map = pd.read_csv(
    '../data/species_split_counts_in_original_manual.csv',
    keep_default_na=False,
    na_values=[''],
)
print(f'{len(manual_map)} rows loaded')
manual_map.head(10)

411 rows loaded


,species_split,manual_species,count
0,NaN,NaN,2
1,(sol),sol?,19
2,aborrskinn,aborrskinn,1
3,agn,agn?,1
4,åkerbinda,åkerbinda,1
5,åkerkrassling,åkerkrassling,1
6,åkerpilört,åkerpilört,1
7,al,al,6271
8,albark,al,1
9,älg,älg,25


In [3]:
def split_manual_species(value):
    if pd.isna(value):
        return [pd.NA]
    return [part.strip() for part in value.split(',')]

long_rows = [
    {'species_split': row.species_split, 'manual_species': part, 'count': row.count}
    for row in manual_map.itertuples(index=False)
    for part in split_manual_species(row.manual_species)
]
manual_long = pd.DataFrame(long_rows)
print(f'{len(manual_map)} manual-mapping rows -> {len(manual_long)} rows after comma-splitting')
manual_long[manual_long['species_split'].astype(str).isin(['bröd- kubbvete', 'enmöjl gran'])]

411 manual-mapping rows -> 415 rows after comma-splitting


,species_split,manual_species,count
50,bröd- kubbvete,brödvete,3
51,bröd- kubbvete,kubbvete,3
106,enmöjl gran,en,0
107,enmöjl gran,gran,0


In [4]:
manual_species_count = (
    manual_long.groupby('manual_species', dropna=False)['count']
    .sum()
    .reset_index()
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
print(f'{len(manual_species_count)} distinct manual_species values')
print(f"total count: {manual_species_count['count'].sum()} (source manual csv totals "
      f"{manual_map['count'].sum()}; higher here since comma-split rows count toward every part)")
manual_species_count.to_csv(OUTPUT_DIR / 'manual_species_count.csv', index=False)
manual_species_count

237 distinct manual_species values
total count: 31379 (source manual csv totals 31374; higher here since comma-split rows count toward every part)


,manual_species,count
0,al,6281
1,tall,3046
2,korn,2230
3,björk,2082
4,ko,1890
...,...,...
232,äppelkärna,1
233,äpple,1
234,åkerbinda,1
235,åkerkrassling,1


## 2. Carry over existing GBIF/SEAD match info

Left-join `manual_long` (the `species_split` → `manual_species` long table from step 1) onto the
old `species_split_taxa_gbif_matches.csv` on the **original** `species_split` key, then collapse to
one row per `manual_species`. Several old `species_split` tokens can map to the same
`manual_species` (e.g. `al`, `albark`, `alknopp`, `alkottar`, `alkotte` → `al`) — where more than
one contributor has a match, prefer the one with a non-null `gbif_usage_key`, and print a
diagnostic if contributors disagree so it can be sanity-checked rather than silently resolved.

In [5]:
old_matches = pd.read_csv('../output/species_split_taxa_gbif_matches.csv')

MATCH_COLS = [
    'match_level', 'genus', 'family', 'order', 'sead_common_name', 'sead_species_name',
    'kingdom', 'gbif_usage_key', 'gbif_canonical_name', 'gbif_rank', 'gbif_match_type',
    'gbif_confidence', 'gbif_url', 'species_split_english',
]

carried = manual_long.merge(old_matches, on='species_split', how='left')

def pick_best(group):
    with_gbif = group.dropna(subset=['gbif_usage_key'])
    conflict = with_gbif['gbif_usage_key'].nunique() > 1
    best = (with_gbif if len(with_gbif) else group).iloc[0]
    contributors = sorted(group['species_split'].dropna().astype(str).unique())
    return pd.Series({
        **best[MATCH_COLS].to_dict(),
        'gbif_conflict': conflict,
        'contributing_species_split': ', '.join(contributors),
    })

carried_best = (
    carried.groupby('manual_species', dropna=False, group_keys=False)
    .apply(pick_best, include_groups=False)
    .reset_index()
)

conflicts = carried_best[carried_best['gbif_conflict']]
print(f'{len(conflicts)} manual_species values have contributors with disagreeing GBIF matches:')
conflicts[['manual_species', 'contributing_species_split', 'gbif_canonical_name']]

1 manual_species values have contributors with disagreeing GBIF matches:


,manual_species,contributing_species_split,gbif_canonical_name
88,hjortdjur,"hjort, hjortdjur, hjorthår",Cervus elaphus


In [6]:
manual_species_taxa = manual_species_count.merge(
    carried_best.drop(columns='gbif_conflict'), on='manual_species', how='left'
)
print(f"{manual_species_taxa['gbif_usage_key'].notna().sum()} of {len(manual_species_taxa)} "
      f"manual_species values carried a GBIF match over from the old file")
manual_species_taxa.to_csv(OUTPUT_DIR / 'manual_species_taxa_gbif_matches.csv', index=False)
manual_species_taxa.sort_values('count', ascending=False).head(20)

142 of 237 manual_species values carried a GBIF match over from the old file


,manual_species,count,match_level,genus,family,order,sead_common_name,sead_species_name,kingdom,gbif_usage_key,gbif_canonical_name,gbif_rank,gbif_match_type,gbif_confidence,gbif_url,species_split_english,contributing_species_split
0,al,6281,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"al, albark, alknopp, alkottar, alkotte"
1,tall,3046,species (common name),Pinus,Pinaceae,Pinales,tall,Pinus sylvestris var sylvestris,Plantae,7230693.0,Pinus sylvestris sylvestris,VARIETY,EXACT,100.0,https://www.gbif.org/species/7230693,Baltic pine,"kottefjäll tall, kottefjäll. tall, tall, tallb..."
2,korn,2230,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"korn, skalkorn"
3,björk,2082,genus (suffix-inferred),Betula,Betulaceae,Fagales,NaN,NaN,Plantae,2875008.0,Betula,GENUS,HIGHERRANK,95.0,https://www.gbif.org/species/2875008,Birch,"björk, björk bulk, björkl, björknäver"
4,ko,1890,animal (common name),NaN,NaN,NaN,kalv,Bos taurus,Animalia,2441022.0,Bos taurus,SPECIES,EXACT,100.0,https://www.gbif.org/species/2441022,Aurochs,"kalv, ko, ötkreatur"
5,ek,1809,species (common name),Quercus,Fagaceae,Fagales,ek,Quercus robur,Plantae,2878688.0,Quercus robur,SPECIES,EXACT,99.0,https://www.gbif.org/species/2878688,Common Oak,"ek, ek bulk, ekbark, ekl, ekollon"
6,hassel,1712,species (common name),Corylus,Corylaceae,Fagales,hassel,Corylus avellana,Plantae,2875979.0,Corylus avellana,SPECIES,EXACT,99.0,https://www.gbif.org/species/2875979,Barcelona-nuts,"hasel, hassel, hasselnöt, hasselskal, obränd h..."
7,gran,1258,species (common name),Picea,Pinaceae,Pinales,gran,Picea abies ssp abies,Plantae,7306267.0,Picea abies abies,SUBSPECIES,EXACT,100.0,https://www.gbif.org/species/7306267,NaN,"enmöjl gran, förkolnade granbarr, gran, granba..."
8,ran,1183,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ran
9,människa,1173,animal (common name),NaN,NaN,NaN,männinska,Homo sapiens,Animalia,2436436.0,Homo sapiens,SPECIES,EXACT,100.0,https://www.gbif.org/species/2436436,Human,"männinska, människa, männska"


## 3. Direct SEAD match on the corrected spelling

Same taxonomy lookups and matching approach as `species_study.ipynb` (`tbl_taxa_common_names` for
Swedish vernacular names, `tbl_taxa_tree_genera`/`_families`/`_orders` for Latin names, a
hand-built `ANIMAL_LATIN` dict for the mammals/fish/seals SEAD has no genera for, and a
suffix-inference fallback for Swedish compound tree names) — but run directly against the
*corrected* `manual_species` spelling instead of the old raw tokens. A typo like `bjrök` never
matched anything before; its correction `björk` should now match straight through the common-name
table. Run **before** any GBIF network calls, since a SEAD match is the more authoritative source
when both agree.

In [7]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv('../.env')

DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.environ["DB_PORT"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [8]:
sv_common = pd.read_sql(
    """
    select lower(cn.common_name) as common_name_lc, cn.common_name,
           v.taxon_id, v.species, v.genus, v.family, v."order"
    from public.tbl_taxa_common_names cn
    join public.view_taxa_alphabetically v on v.taxon_id = cn.taxon_id
    where cn.language_id = 2
    """,
    engine,
)
genera = pd.read_sql(
    'select lower(genus_name) as genus_lc, genus_name, genus_id, family_id from public.tbl_taxa_tree_genera', engine
)
families = pd.read_sql(
    'select lower(family_name) as family_lc, family_name, family_id, order_id from public.tbl_taxa_tree_families', engine
)
orders_lookup = pd.read_sql(
    'select lower(order_name) as order_lc, order_name, order_id from public.tbl_taxa_tree_orders', engine
)
# Raw taxonomy tables kept around (not just the *_hierarchy lookups below) so step 5 can find
# existing "indet" placeholders and compute next-available IDs for proposed new records.
taxa_master = pd.read_sql('select taxon_id, genus_id, species from public.tbl_taxa_tree_master', engine)
common_names_all = pd.read_sql(
    'select taxon_common_name_id, common_name, taxon_id, language_id from public.tbl_taxa_common_names', engine
)

common_map = sv_common.drop_duplicates('common_name_lc').set_index('common_name_lc')
genus_hierarchy = (
    genera.merge(families[['family_id', 'family_name', 'order_id']], on='family_id', how='left')
          .merge(orders_lookup[['order_id', 'order_name']], on='order_id', how='left')
          .drop_duplicates('genus_lc').set_index('genus_lc')
)
family_hierarchy = (
    families.merge(orders_lookup[['order_id', 'order_name']], on='order_id', how='left')
             .drop_duplicates('family_lc').set_index('family_lc')
)
order_hierarchy = orders_lookup.drop_duplicates('order_lc').set_index('order_lc')

print(f'{len(sv_common)} Swedish common names, {len(genus_hierarchy)} genera, '
      f'{len(family_hierarchy)} families, {len(order_hierarchy)} orders, {len(taxa_master)} taxa loaded')

4272 Swedish common names, 5212 genera, 541 families, 57 orders, 23981 taxa loaded


In [9]:
# Same manual dictionary as species_study.ipynb: SEAD's taxonomy has no mammal/fish/seal genera
# at all (confirmed there), so these domesticated/wild animal Swedish names are matched by hand
# to their Latin name/rank instead of relying on tbl_taxa_common_names.
ANIMAL_LATIN = {
    'ko': ('species', 'Bos taurus'), 'nöt': ('species', 'Bos taurus'),
    'nötboskap': ('species', 'Bos taurus'), 'nötkreatur': ('species', 'Bos taurus'),
    'nötkrestur': ('species', 'Bos taurus'), 'nörkreatur': ('species', 'Bos taurus'),
    'ötkreatur': ('species', 'Bos taurus'), 'kalv': ('species', 'Bos taurus'),
    'uroxe': ('species', 'Bos primigenius'),
    'svin': ('species', 'Sus scrofa domesticus'), 'tamsvin': ('species', 'Sus scrofa domesticus'),
    'gris': ('species', 'Sus scrofa domesticus'), 'vildsvin': ('species', 'Sus scrofa'),
    'get': ('species', 'Capra hircus'), 'får': ('species', 'Ovis aries'),
    'lamm': ('species', 'Ovis aries'), 'häst': ('species', 'Equus caballus'),
    'hund': ('species', 'Canis lupus familiaris'), 'katt': ('species', 'Felis catus'),
    'höna': ('species', 'Gallus gallus domesticus'),
    'älg': ('species', 'Alces alces'), 'ren': ('species', 'Rangifer tarandus'),
    'rådjur': ('species', 'Capreolus capreolus'), 'hjort': ('species', 'Cervus elaphus'),
    'kronhjort': ('species', 'Cervus elaphus'), 'räv': ('species', 'Vulpes vulpes'),
    'björn': ('species', 'Ursus arctos'), 'bäver': ('species', 'Castor fiber'),
    'gädda': ('species', 'Esox lucius'), 'torsk': ('species', 'Gadus morhua'),
    'sik': ('species', 'Coregonus lavaretus'), 'mört': ('species', 'Rutilus rutilus'),
    'aborrskinn': ('species', 'Perca fluviatilis'),
    'gråsäl': ('species', 'Halichoerus grypus'), 'grönlandssäl': ('species', 'Pagophilus groenlandicus'),
    'vikare': ('species', 'Pusa hispida'), 'vikaresäl': ('species', 'Pusa hispida'),
    'ostron': ('species', 'Ostrea edulis'), 'kärrsköldpadda': ('species', 'Emys orbicularis'),
    'männinska': ('species', 'Homo sapiens'), 'människa': ('species', 'Homo sapiens'),
    'männska': ('species', 'Homo sapiens'),
    'delfin': ('family', 'Delphinidae'), 'säl': ('family', 'Phocidae'),
    'hjortdjur': ('family', 'Cervidae'), 'gnagare': ('order', 'Rodentia'),
}

def match_animal(value):
    if value not in ANIMAL_LATIN:
        return None
    rank, latin = ANIMAL_LATIN[value]
    if rank == 'species':
        return dict(match_level='animal (common name)', genus=None, family=None, order=None,
                    sead_common_name=value, sead_species_name=latin, kingdom='Animalia', taxon_id=None)
    if rank == 'family':
        return dict(match_level='animal (family)', genus=None, family=latin, order=None,
                    sead_common_name=value, sead_species_name=None, kingdom='Animalia', taxon_id=None)
    return dict(match_level='animal (order)', genus=None, family=None, order=latin,
                sead_common_name=value, sead_species_name=None, kingdom='Animalia', taxon_id=None)

# Non-specific animal/plant-material terms the suffix-inference heuristic would otherwise mis-assign
# to an unrelated genus purely by spelling coincidence (see species_study.ipynb for the full list of
# examples, e.g. 'horn' -> Ibicella, 'läder' -> Sambucus via 'fläder').
NON_TAXONOMIC_BLOCKLIST = {'horn', 'hår', 'läder', 'bröd', 'spel'}

LATIN_QUALIFIER_RE = re.compile(r'^(cf\.?\s+|aff\.?\s+)|(\s+(sp\.?|spp\.?|indet\.?)\s*$)', re.I)

def strip_latin_qualifiers(value):
    prev = None
    while prev != value:
        prev = value
        value = LATIN_QUALIFIER_RE.sub('', value).strip()
    return value

def match_exact(value_lc):
    if value_lc in common_map.index:
        rec = common_map.loc[value_lc]
        return dict(match_level='species (common name)', genus=rec['genus'], family=rec['family'], order=rec['order'],
                    sead_common_name=rec['common_name'], sead_species_name=f"{rec['genus']} {rec['species']}",
                    kingdom='Plantae', taxon_id=int(rec['taxon_id']))
    cleaned = strip_latin_qualifiers(value_lc)
    if cleaned in genus_hierarchy.index:
        rec = genus_hierarchy.loc[cleaned]
        return dict(match_level='genus (latin)', genus=rec['genus_name'], family=rec['family_name'], order=rec['order_name'],
                    sead_common_name=None, sead_species_name=None, kingdom='Plantae', taxon_id=None)
    if cleaned in family_hierarchy.index:
        rec = family_hierarchy.loc[cleaned]
        return dict(match_level='family (latin)', genus=None, family=rec['family_name'], order=rec['order_name'],
                    sead_common_name=None, sead_species_name=None, kingdom='Plantae', taxon_id=None)
    if cleaned in order_hierarchy.index:
        rec = order_hierarchy.loc[cleaned]
        return dict(match_level='order (latin)', genus=None, family=None, order=rec['order_name'],
                    sead_common_name=None, sead_species_name=None, kingdom='Plantae', taxon_id=None)
    return None

def infer_genus_by_suffix(value_lc):
    """Swedish tree names compound as <modifier><base>, e.g. 'klibbal' -> Alnus. Only accepted
    when every common name ending in the value converges on a single genus."""
    if len(value_lc) < 2 or not value_lc.isalpha():
        return None
    hits = sv_common[sv_common['common_name_lc'].str.endswith(value_lc)]
    if hits.empty:
        return None
    candidate_genera = hits['genus'].unique().tolist()
    if len(candidate_genera) == 1:
        rec = genus_hierarchy[genus_hierarchy['genus_name'] == candidate_genera[0]]
        return dict(match_level='genus (suffix-inferred)', genus=candidate_genera[0],
                    family=rec['family_name'].iloc[0] if len(rec) else None,
                    order=rec['order_name'].iloc[0] if len(rec) else None,
                    sead_common_name=None, sead_species_name=None, kingdom='Plantae', taxon_id=None)
    return dict(match_level='genus (suffix, ambiguous)', genus=None, family=None, order=None,
                sead_common_name=None, sead_species_name=None, kingdom=None, taxon_id=None,
                ambiguous_genus_candidates=', '.join(candidate_genera))

def direct_sead_match(value):
    """value is the raw manual_species text (may end in '?'); matching itself is done on the
    cleaned lowercase form, with the '?' stripped only for lookup purposes, not for output."""
    value_lc = value.rstrip('?').strip().lower()
    if value_lc in NON_TAXONOMIC_BLOCKLIST:
        return None
    return match_animal(value_lc) or match_exact(value_lc) or infer_genus_by_suffix(value_lc)

In [10]:
def apply_direct_match(manual_species):
    if pd.isna(manual_species):
        return pd.Series(dtype=object)
    result = direct_sead_match(manual_species)
    return pd.Series(result) if result else pd.Series(dtype=object)

direct = manual_species_taxa['manual_species'].apply(apply_direct_match)
direct = direct.rename(columns={c: f'direct_{c}' for c in direct.columns})
print(f"direct SEAD match found something for {direct['direct_match_level'].notna().sum() if 'direct_match_level' in direct else 0} "
      f"of {len(manual_species_taxa)} manual_species values")

manual_species_taxa = pd.concat([manual_species_taxa, direct], axis=1)

# Where the direct match disagrees with the carried-over match_level, log it for a sanity check,
# then let the direct match (running against the corrected spelling) win.
disagreements = manual_species_taxa[
    manual_species_taxa['direct_match_level'].notna()
    & manual_species_taxa['match_level'].notna()
    & (manual_species_taxa['direct_match_level'] != manual_species_taxa['match_level'])
]
print(f'{len(disagreements)} rows where the direct match disagrees with the carried-over match_level:')
disagreements[['manual_species', 'match_level', 'direct_match_level', 'genus', 'direct_genus']]

direct SEAD match found something for 156 of 237 manual_species values


1 rows where the direct match disagrees with the carried-over match_level:


,manual_species,match_level,direct_match_level,genus,direct_genus
47,hjortdjur,animal (common name),animal (family),NaN,NaN


In [11]:
# The direct match (on the corrected spelling) wins over whatever step 2 carried from the old,
# typo-riddled tokens whenever it found something.
OVERRIDE_COLS = ['match_level', 'genus', 'family', 'order', 'sead_common_name', 'sead_species_name', 'kingdom']
for col in OVERRIDE_COLS:
    direct_col = f'direct_{col}'
    manual_species_taxa[col] = manual_species_taxa[direct_col].where(
        manual_species_taxa[direct_col].notna(), manual_species_taxa[col]
    )

manual_species_taxa['taxon_id'] = manual_species_taxa['direct_taxon_id']
manual_species_taxa['ambiguous_genus_candidates'] = manual_species_taxa.get('direct_ambiguous_genus_candidates')
manual_species_taxa = manual_species_taxa.drop(columns=[c for c in manual_species_taxa.columns if c.startswith('direct_')])

print(f"{manual_species_taxa['match_level'].notna().sum()} of {len(manual_species_taxa)} rows now have "
      f"a match_level (carried-over or direct)")
manual_species_taxa['match_level'].value_counts(dropna=False)

159 of 237 rows now have a match_level (carried-over or direct)


match_level
NaN                          78
species (common name)        64
genus (latin)                33
animal (common name)         32
genus (suffix, ambiguous)    16
genus (suffix-inferred)      10
animal (family)               3
animal (order)                1
Name: count, dtype: int64

## 4. Fill remaining GBIF gaps

Two passes, both only touching rows that still have no `gbif_usage_key`:

1. **Candidate cross-check** — for any row that now has a SEAD-resolved Latin name (species
   binomial, else genus, else family, else order — from carryover *or* the direct match above),
   query GBIF's `species/match` with that Latin name, exactly as `species_study.ipynb` did. This
   is the majority of rows still missing GBIF info, since step 3 resolved several that step 2's
   carryover missed.
2. **Free-text fallback** — for rows with *no* Latin anchor at all (nothing from SEAD either),
   try GBIF's general `species/search?q=<term>` directly on the Swedish word, in case it turns up
   a plausible vernacular-name hit. Most of these are expected to come back empty (they're
   artifact/material categories like `amulettring`, `stål`, `kalkbruk`, not organisms), which is
   fine — no special-casing needed.

In [12]:
import requests

GBIF_SESSION = requests.Session()

def gbif_match(name, kingdom):
    try:
        params = {'name': name}
        if pd.notna(kingdom):
            params['kingdom'] = kingdom
        response = GBIF_SESSION.get('https://api.gbif.org/v1/species/match', params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException:
        return None
    if data.get('matchType') in (None, 'NONE'):
        return None
    return {
        'gbif_usage_key': data.get('usageKey'),
        'gbif_canonical_name': data.get('canonicalName'),
        'gbif_rank': data.get('rank'),
        'gbif_match_type': data.get('matchType'),
        'gbif_confidence': data.get('confidence'),
        'gbif_source': 'species/match',
        'gbif_genus': data.get('genus'),
        'gbif_family': data.get('family'),
        'gbif_order': data.get('order'),
        'gbif_kingdom': data.get('kingdom'),
    }

def gbif_vernacular_search(term):
    """Fallback for terms with no SEAD-resolved Latin anchor: GBIF's general search also matches
    against indexed vernacular names, so a plain Swedish word can still turn up a plausible hit."""
    try:
        response = GBIF_SESSION.get(
            'https://api.gbif.org/v1/species/search',
            params={'q': term, 'status': 'ACCEPTED', 'limit': 5},
            timeout=10,
        )
        response.raise_for_status()
        results = response.json().get('results', [])
    except requests.RequestException:
        return None
    # Only accept a result that actually lists the query term among its own vernacular names -
    # GBIF's free-text search matches scientific names too, which would otherwise let an
    # unrelated Latin-looking coincidence through.
    for result in results:
        vernaculars = {v.get('vernacularName', '').lower() for v in result.get('vernacularNames', [])}
        if term.lower() in vernaculars:
            return {
                'gbif_usage_key': result.get('key'),
                'gbif_canonical_name': result.get('canonicalName') or result.get('scientificName'),
                'gbif_rank': result.get('rank'),
                'gbif_match_type': 'VERNACULAR',
                'gbif_confidence': None,
                'gbif_source': 'species/search (vernacular)',
                'gbif_genus': result.get('genus'),
                'gbif_family': result.get('family'),
                'gbif_order': result.get('order'),
                'gbif_kingdom': result.get('kingdom'),
            }
    return None

GBIF_FIELDS = [
    'gbif_usage_key', 'gbif_canonical_name', 'gbif_rank', 'gbif_match_type', 'gbif_confidence',
    'gbif_source', 'gbif_genus', 'gbif_family', 'gbif_order', 'gbif_kingdom',
]

In [13]:
# Mark existing (carried-over) GBIF matches with their source, so gbif_source ends up populated
# for every row that has a gbif_usage_key, whichever pass it came from. Also make sure the newer
# GBIF_FIELDS (genus/family/order/kingdom) exist as columns even though the carried-over old file
# never had them.
for field in GBIF_FIELDS:
    if field not in manual_species_taxa.columns:
        manual_species_taxa[field] = pd.NA
manual_species_taxa.loc[
    manual_species_taxa['gbif_usage_key'].notna() & manual_species_taxa['gbif_source'].isna(), 'gbif_source'
] = 'carried-over (species_split_taxa_gbif_matches.csv)'

def gbif_lookup_by_key(usage_key):
    try:
        response = GBIF_SESSION.get(f'https://api.gbif.org/v1/species/{int(usage_key)}', timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException:
        return None
    return {'gbif_genus': data.get('genus'), 'gbif_family': data.get('family'),
            'gbif_order': data.get('order'), 'gbif_kingdom': data.get('kingdom')}

# The old species_split_taxa_gbif_matches.csv (carried over in step 2) never had genus/family/order
# columns, so rows that already carried a gbif_usage_key from there never went through the
# candidate cross-check below and are missing that breakdown - backfill it with one lookup per
# distinct usage_key rather than skipping these rows for the rest of step 4/5.
needs_backfill = manual_species_taxa[
    manual_species_taxa['gbif_usage_key'].notna() & manual_species_taxa['gbif_genus'].isna()
]
backfill_cache = {key: gbif_lookup_by_key(key) for key in needs_backfill['gbif_usage_key'].unique()}
backfilled = manual_species_taxa['gbif_usage_key'].map(backfill_cache)
for field in ['gbif_genus', 'gbif_family', 'gbif_order', 'gbif_kingdom']:
    manual_species_taxa[field] = manual_species_taxa[field].fillna(backfilled.apply(
        lambda d: d.get(field) if isinstance(d, dict) else None
    ))
print(f'{len(backfill_cache)} carried-over GBIF matches backfilled with genus/family/order')

# Best Latin candidate to cross-check against GBIF: species binomial if resolved, else genus,
# else family, else order - from either the step 2 carryover or the step 3 direct match.
manual_species_taxa['gbif_candidate'] = (
    manual_species_taxa['sead_species_name']
    .fillna(manual_species_taxa['genus'])
    .fillna(manual_species_taxa['family'])
    .fillna(manual_species_taxa['order'])
)

needs_gbif = manual_species_taxa[
    manual_species_taxa['gbif_usage_key'].isna() & manual_species_taxa['gbif_candidate'].notna()
]
candidate_kingdom = needs_gbif.drop_duplicates('gbif_candidate').set_index('gbif_candidate')['kingdom']

gbif_cache = {
    candidate: gbif_match(candidate, candidate_kingdom.get(candidate))
    for candidate in needs_gbif['gbif_candidate'].dropna().unique()
}
resolved = sum(1 for v in gbif_cache.values() if v)
print(f'{resolved} of {len(gbif_cache)} new SEAD-resolved candidates matched in GBIF')

124 carried-over GBIF matches backfilled with genus/family/order
1 of 1 new SEAD-resolved candidates matched in GBIF


In [14]:
def apply_gbif_result(row):
    if pd.notna(row['gbif_usage_key']):
        return pd.Series({f: row.get(f) for f in GBIF_FIELDS})
    result = gbif_cache.get(row['gbif_candidate'])
    if not result:
        return pd.Series({f: pd.NA for f in GBIF_FIELDS})
    return pd.Series(result)

updated = manual_species_taxa.apply(apply_gbif_result, axis=1)
manual_species_taxa[GBIF_FIELDS] = updated[GBIF_FIELDS]

print(f"{manual_species_taxa['gbif_usage_key'].notna().sum()} of {len(manual_species_taxa)} rows have a "
      f"GBIF match after the candidate cross-check")
manual_species_taxa[manual_species_taxa['gbif_source'] == 'species/match'][
    ['manual_species', 'gbif_candidate', 'gbif_canonical_name', 'gbif_match_type', 'gbif_confidence']
]

143 of 237 rows have a GBIF match after the candidate cross-check


,manual_species,gbif_candidate,gbif_canonical_name,gbif_match_type,gbif_confidence
166,cerealia?,Cerealia,Plantae,HIGHERRANK,99


In [15]:
still_missing = manual_species_taxa[
    manual_species_taxa['gbif_usage_key'].isna()
    & manual_species_taxa['gbif_candidate'].isna()
    & manual_species_taxa['manual_species'].notna()
]
vernacular_cache = {
    term: gbif_vernacular_search(term.rstrip('?').strip())
    for term in still_missing['manual_species'].unique()
}
found_vern = sum(1 for v in vernacular_cache.values() if v)
print(f'{found_vern} of {len(vernacular_cache)} SEAD-unresolved terms found an exact vernacular-name '
      f'hit via GBIF free-text search')

def apply_vernacular_result(row):
    if pd.notna(row['gbif_usage_key']) or row['manual_species'] not in vernacular_cache:
        return pd.Series({f: row.get(f) for f in GBIF_FIELDS})
    result = vernacular_cache.get(row['manual_species'])
    if not result:
        return pd.Series({f: row.get(f) for f in GBIF_FIELDS})
    return pd.Series(result)

updated = manual_species_taxa.apply(apply_vernacular_result, axis=1)
manual_species_taxa[GBIF_FIELDS] = updated[GBIF_FIELDS]

print(f"{manual_species_taxa['gbif_usage_key'].notna().sum()} of {len(manual_species_taxa)} rows have a "
      f"GBIF match after both passes")
manual_species_taxa[manual_species_taxa['gbif_source'] == 'species/search (vernacular)']

16 of 93 SEAD-unresolved terms found an exact vernacular-name hit via GBIF free-text search
159 of 237 rows have a GBIF match after both passes


,manual_species,count,match_level,genus,family,order,sead_common_name,sead_species_name,kingdom,gbif_usage_key,gbif_canonical_name,gbif_rank,gbif_match_type,gbif_confidence,gbif_url,species_split_english,contributing_species_split,taxon_id,ambiguous_genus_candidates,gbif_source,gbif_genus,gbif_family,gbif_order,gbif_kingdom,gbif_candidate
36,nävesläktet?,68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,304058091,Geranium,GENUS,VERNACULAR,None,NaN,NaN,näver,NaN,NaN,species/search (vernacular),Geranium,Geraniaceae,Geraniales,Plantae,NaN
39,kubbvete,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,160025822,Triticum compactum,SPECIES,VERNACULAR,None,NaN,NaN,"bröd- kubbvete, kubbvete",NaN,NaN,species/search (vernacular),Triticum,Poaceae,Poales,Plantae,NaN
53,djur,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,159935719,Animalia,KINGDOM,VERNACULAR,None,NaN,NaN,djurben,NaN,NaN,species/search (vernacular),None,None,None,Animalia,NaN
57,ull,14,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,NaN,160027178,Eriophorum,GENUS,VERNACULAR,None,NaN,NaN,"ull, ulltextil",NaN,"Nymphoides, Polemonium, Eriophorum",species/search (vernacular),Eriophorum,Cyperaceae,Poales,Plantae,NaN
61,Starrsläktet,12,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,NaN,296322706,Carex,GENUS,VERNACULAR,None,NaN,NaN,starr,NaN,NaN,species/search (vernacular),Carex,Cyperaceae,Poales,Plantae,NaN
107,gråärt,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,160020585,Pisum sativum arvense,VARIETY,VERNACULAR,None,NaN,NaN,gråärt,NaN,NaN,species/search (vernacular),Pisum,Fabaceae,Fabales,Plantae,NaN
127,idisslare,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,159939909,Ruminantia,SUBORDER,VERNACULAR,None,NaN,NaN,idisslare,NaN,NaN,species/search (vernacular),None,None,Artiodactyla,Animalia,NaN
149,bergssyra,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,160025141,Rumex acetosella,SPECIES,VERNACULAR,None,NaN,NaN,bergssyra,NaN,NaN,species/search (vernacular),Rumex,Polygonaceae,Caryophyllales,Plantae,NaN
157,dån,1,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,NaN,160016506,Galeopsis,GENUS,VERNACULAR,None,NaN,NaN,dån,NaN,"Galeopsis, Leucas",species/search (vernacular),Galeopsis,Lamiaceae,Lamiales,Plantae,NaN
158,dinkel,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,159501969,Triticum spelta,SPECIES,VERNACULAR,None,NaN,NaN,dinkel,NaN,NaN,species/search (vernacular),Triticum,Poaceae,Poales,Plantae,NaN


In [16]:
# Where nothing anchors this row to a rank yet (true empty match, or a still-unresolved ambiguous
# genus) but GBIF's free-text vernacular search found something, use GBIF's own genus/family/order
# as the taxonomic anchor going into step 5 - still checked against SEAD's tables there, not
# assumed to already exist in SEAD. match_level text is left as-is (e.g. still says "genus
# (suffix, ambiguous)") so step 5 can keep flagging those for review even once anchored.
no_sead_anchor = (
    manual_species_taxa['genus'].isna() & manual_species_taxa['family'].isna()
    & manual_species_taxa['order'].isna() & manual_species_taxa['gbif_usage_key'].notna()
)
manual_species_taxa.loc[no_sead_anchor, 'genus'] = manual_species_taxa.loc[no_sead_anchor, 'gbif_genus']
manual_species_taxa.loc[no_sead_anchor, 'family'] = manual_species_taxa.loc[no_sead_anchor, 'gbif_family']
manual_species_taxa.loc[no_sead_anchor, 'order'] = manual_species_taxa.loc[no_sead_anchor, 'gbif_order']
manual_species_taxa.loc[no_sead_anchor, 'kingdom'] = manual_species_taxa.loc[no_sead_anchor, 'gbif_kingdom']
manual_species_taxa.loc[no_sead_anchor & manual_species_taxa['match_level'].isna(), 'match_level'] = (
    'gbif only (no prior SEAD anchor)'
)
print(f'{no_sead_anchor.sum()} rows got a taxonomic anchor purely from GBIF (no SEAD-resolved rank beforehand)')
manual_species_taxa.loc[no_sead_anchor, ['manual_species', 'match_level', 'gbif_canonical_name', 'genus', 'family', 'order']]

48 rows got a taxonomic anchor purely from GBIF (no SEAD-resolved rank beforehand)


,manual_species,match_level,gbif_canonical_name,genus,family,order
4,ko,animal (common name),Bos taurus,Bos,Bovidae,Artiodactyla
9,människa,animal (common name),Homo sapiens,Homo,Hominidae,Primates
14,däggdjur?,animal (common name),Bos taurus,Bos,Bovidae,Artiodactyla
32,häst,animal (common name),Equus caballus,Equus,Equidae,Perissodactyla
34,svin,animal (common name),Sus scrofa domesticus,Sus,Suidae,Artiodactyla
36,nävesläktet?,gbif only (no prior SEAD anchor),Geranium,Geranium,Geraniaceae,Geraniales
38,får,animal (common name),Ovis aries,Ovis,Bovidae,Artiodactyla
39,kubbvete,gbif only (no prior SEAD anchor),Triticum compactum,Triticum,Poaceae,Poales
43,get,animal (common name),Capra hircus,Capra,Bovidae,Artiodactyla
49,älg,animal (common name),Alces alces,Alces,Cervidae,Artiodactyla


In [17]:
def gbif_english_name(usage_key):
    try:
        response = GBIF_SESSION.get(
            f'https://api.gbif.org/v1/species/{int(usage_key)}/vernacularNames', params={'limit': 50}, timeout=10
        )
        response.raise_for_status()
        results = response.json().get('results', [])
    except requests.RequestException:
        return pd.NA
    english_names = [r['vernacularName'] for r in results if r.get('language') in ('eng', 'en')]
    return english_names[0] if english_names else pd.NA

missing_english = manual_species_taxa[
    manual_species_taxa['gbif_usage_key'].notna() & manual_species_taxa['species_split_english'].isna()
]
english_cache = {key: gbif_english_name(key) for key in missing_english['gbif_usage_key'].unique()}
manual_species_taxa['species_split_english'] = manual_species_taxa['species_split_english'].fillna(
    manual_species_taxa['gbif_usage_key'].map(english_cache)
)
print(f"{manual_species_taxa['species_split_english'].notna().sum()} of {len(manual_species_taxa)} rows "
      f"have an English name")

manual_species_taxa.to_csv(OUTPUT_DIR / 'manual_species_taxa_gbif_matches.csv', index=False)
manual_species_taxa.sort_values('count', ascending=False).head(15)

136 of 237 rows have an English name


,manual_species,count,match_level,genus,family,order,sead_common_name,sead_species_name,kingdom,gbif_usage_key,gbif_canonical_name,gbif_rank,gbif_match_type,gbif_confidence,gbif_url,species_split_english,contributing_species_split,taxon_id,ambiguous_genus_candidates,gbif_source,gbif_genus,gbif_family,gbif_order,gbif_kingdom,gbif_candidate
0,al,6281,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,"al, albark, alknopp, alkottar, alkotte",NaN,"Lathyrus, Alnus",<NA>,<NA>,<NA>,<NA>,<NA>,NaN
1,tall,3046,species (common name),Pinus,Pinaceae,Pinales,tall,Pinus sylvestris var sylvestris,Plantae,7230693.0,Pinus sylvestris sylvestris,VARIETY,EXACT,100.0,https://www.gbif.org/species/7230693,Baltic pine,"kottefjäll tall, kottefjäll. tall, tall, tallb...",3613.0,NaN,carried-over (species_split_taxa_gbif_matches....,Pinus,Pinaceae,Pinales,Plantae,Pinus sylvestris var sylvestris
2,korn,2230,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,"korn, skalkorn",NaN,"Impatiens, Hordelymus, Hordeum",<NA>,<NA>,<NA>,<NA>,<NA>,NaN
3,björk,2082,genus (suffix-inferred),Betula,Betulaceae,Fagales,NaN,NaN,Plantae,2875008.0,Betula,GENUS,HIGHERRANK,95.0,https://www.gbif.org/species/2875008,Birch,"björk, björk bulk, björkl, björknäver",NaN,NaN,carried-over (species_split_taxa_gbif_matches....,Betula,Betulaceae,Fagales,Plantae,Betula
4,ko,1890,animal (common name),Bos,Bovidae,Artiodactyla,ko,Bos taurus,Animalia,2441022.0,Bos taurus,SPECIES,EXACT,100.0,https://www.gbif.org/species/2441022,Aurochs,"kalv, ko, ötkreatur",NaN,NaN,carried-over (species_split_taxa_gbif_matches....,Bos,Bovidae,Artiodactyla,Animalia,Bos taurus
5,ek,1809,species (common name),Quercus,Fagaceae,Fagales,ek,Quercus robur,Plantae,2878688.0,Quercus robur,SPECIES,EXACT,99.0,https://www.gbif.org/species/2878688,Common Oak,"ek, ek bulk, ekbark, ekl, ekollon",2614.0,NaN,carried-over (species_split_taxa_gbif_matches....,Quercus,Fagaceae,Fagales,Plantae,Quercus robur
6,hassel,1712,species (common name),Corylus,Corylaceae,Fagales,hassel,Corylus avellana,Plantae,2875979.0,Corylus avellana,SPECIES,EXACT,99.0,https://www.gbif.org/species/2875979,Barcelona-nuts,"hasel, hassel, hasselnöt, hasselskal, obränd h...",1825.0,NaN,carried-over (species_split_taxa_gbif_matches....,Corylus,Betulaceae,Fagales,Plantae,Corylus avellana
7,gran,1258,species (common name),Picea,Pinaceae,Pinales,gran,Picea abies ssp abies,Plantae,7306267.0,Picea abies abies,SUBSPECIES,EXACT,100.0,https://www.gbif.org/species/7306267,NaN,"enmöjl gran, förkolnade granbarr, gran, granba...",3591.0,NaN,carried-over (species_split_taxa_gbif_matches....,Picea,Pinaceae,Pinales,Plantae,Picea abies ssp abies
8,ran,1183,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,ran,NaN,"Abies, Picea, Pseudotsuga, Taxus",<NA>,<NA>,<NA>,<NA>,<NA>,NaN
9,människa,1173,animal (common name),Homo,Hominidae,Primates,människa,Homo sapiens,Animalia,2436436.0,Homo sapiens,SPECIES,EXACT,100.0,https://www.gbif.org/species/2436436,Human,"männinska, människa, männska",NaN,NaN,carried-over (species_split_taxa_gbif_matches....,Homo,Hominidae,Primates,Animalia,Homo sapiens


## 5. Resolve/propose SEAD `taxon_id` + `common_name_id`

For every row with a taxonomic anchor (species/genus/family/order, from SEAD or GBIF), walk down
from whatever rank is known to a concrete `taxon_id`:

- A `species (common name)` match already carries its `taxon_id` straight from `tbl_taxa_common_names`
  - reuse it as-is (nothing to do, per "match by common name").
- Otherwise, resolve/propose `order_id` -> `family_id` -> `genus_id` (reusing an existing SEAD row
  whenever the Latin name already matches one, case-insensitively; otherwise proposing a new one,
  named `"<parent> indet"` when the rank itself is unknown, following the one existing
  `Cyperaceae indet` precedent).
- If only a genus is known (no specific species - e.g. `cf picea sp`) *and* that genus already has
  several Swedish common names in SEAD tied to different species (Picea alone has 6: gran/abies,
  altaigran/glauca, svartgran/mariana, serbgran/omorika, blagran/pungens - Salix has ~40), there is
  no way to infer from the schema alone which one is "the" generic representative for this
  dataset - so that's a hand-curated lookup (`GENUS_DEFAULT_COMMON_NAME`) rather than something
  guessed at automatically. When the genus isn't in that lookup, resolve/propose a `taxon_id`
  under it instead - reusing an existing `sp.`/`indet.` placeholder taxon if SEAD already has one
  for that genus, otherwise proposing a new one.
- Then resolve/propose a `taxon_common_name_id` for the taxon: reused if it already has a Swedish
  common name, otherwise proposed as new - using the corrected Swedish `manual_species` text where
  that text genuinely *is* a Swedish common name, or GBIF's English vernacular name (with
  `language_id=1`) when `manual_species` is actually a raw Latin genus/family/order designation
  (`match_level` containing `"(latin)"`, e.g. `salix`, `salix sp`, `cf picea sp`) rather than a
  common name in any language.

New IDs continue SEAD's own numbering (`max(existing)+1`) and are deduplicated within this run.
Every row's output also spells out `resolved_order`/`resolved_family`/`resolved_genus`/
`resolved_species` (the actual rank names/species landed on, existing or new, so the resolution is
readable without cross-referencing another table), plus `taxonomy_changes_needed` - a plain-text
summary of exactly what would need to be added to SEAD for this row (e.g. `"new genus 'X indet'
under existing family 'Y'; new taxon 'X indet indet.'; new English common name 'Sallow'"`), and
`matched_existing_taxon` to flag when nothing new was needed at all.

Rows are flagged `needs_manual_review` (not auto-trusted) when: nothing anchors them at all, the
genus-suffix match is still ambiguous even after the GBIF cross-check, GBIF's own match was weak
(`FUZZY`/`VERNACULAR`/low confidence), no Swedish or English text was available to attach as a
common name, or the manual mapping marked the value `?` (uncertain, explicitly meant to stay
questioned per the source data).

In [18]:
next_order_id = int(orders_lookup['order_id'].max()) + 1
next_family_id = int(families['family_id'].max()) + 1
next_genus_id = int(genera['genus_id'].max()) + 1
next_taxon_id = int(taxa_master['taxon_id'].max()) + 1
next_common_name_id = int(common_names_all['taxon_common_name_id'].max()) + 1

LANGUAGE_NAMES = {1: 'English', 2: 'Swedish'}

# Hand-curated: when a manual_species value only resolves to a GENUS (no specific species known,
# e.g. 'cf picea sp' -> genus Picea), and that genus already has more than one Swedish common name
# in SEAD tied to different species (Picea alone has 6: gran/abies, altaigran/glauca,
# svartgran/mariana, serbgran/omorika, blagran/pungens - Salix has ~40), there is no way to infer
# from the schema alone which one is "the" generic representative for this dataset - it has to be
# told to us rather than guessed. Seed it with what's been confirmed; extend as more genus-only
# cases get reviewed. A genus not listed here falls through to a fresh 'sp.'/'indet.' placeholder
# taxon instead (see resolve_taxon) - the safe default when there's no single obvious match.
GENUS_DEFAULT_COMMON_NAME = {
    'Picea': 'gran',
}

# Registries so two rows needing the same new placeholder within this run reuse one proposed
# record instead of creating a duplicate.
proposed_orders = {}         # order_name_lc -> order_id
proposed_families = {}       # family_name_lc -> family_id
proposed_genera = {}         # genus_name_lc -> genus_id
proposed_taxa = {}           # (genus_id, species_lc) -> taxon_id
proposed_common_names = {}   # (taxon_id, language_id, common_name_lc) -> taxon_common_name_id
new_records = []             # rows for new_sead_records.csv
CURRENT_MANUAL_SPECIES = None

def resolve_order(order_name):
    """Returns (order_id, is_new, name_used)."""
    if order_name is None:
        return None, False, None
    lc = order_name.lower()
    if lc in order_hierarchy.index:
        return int(order_hierarchy.loc[lc, 'order_id']), False, order_name
    if lc in proposed_orders:
        return proposed_orders[lc], True, order_name
    global next_order_id
    order_id = next_order_id
    next_order_id += 1
    proposed_orders[lc] = order_id
    new_records.append(dict(table='tbl_taxa_tree_orders', id_column='order_id', id=order_id,
                             name=order_name, language_id=None, parent_table=None, parent_id=None,
                             created_for=CURRENT_MANUAL_SPECIES))
    return order_id, True, order_name

def resolve_family(family_name, order_id, order_name_for_placeholder):
    """order_id must already be resolved by the caller. family_name=None -> propose a
    '<order> indet' placeholder family under order_id (or generic 'Indet' if the order itself is
    unknown too). Returns (family_id, is_new, name_used)."""
    if family_name is None:
        family_name = f'{order_name_for_placeholder} indet' if order_name_for_placeholder else 'Indet'
    lc = family_name.lower()
    if lc in family_hierarchy.index:
        return int(family_hierarchy.loc[lc, 'family_id']), False, family_name
    if lc in proposed_families:
        return proposed_families[lc], True, family_name
    global next_family_id
    family_id = next_family_id
    next_family_id += 1
    proposed_families[lc] = family_id
    new_records.append(dict(table='tbl_taxa_tree_families', id_column='family_id', id=family_id,
                             name=family_name, language_id=None, parent_table='tbl_taxa_tree_orders',
                             parent_id=order_id, created_for=CURRENT_MANUAL_SPECIES))
    return family_id, True, family_name

def resolve_genus(genus_name, family_id, family_name_for_placeholder):
    """family_id must already be resolved by the caller. genus_name=None -> propose a
    '<family> indet' placeholder genus under family_id (matches the one existing SEAD precedent,
    'Cyperaceae indet'). Returns (genus_id, is_new, name_used)."""
    if genus_name is None:
        genus_name = f'{family_name_for_placeholder} indet' if family_name_for_placeholder else 'Indet'
    lc = genus_name.lower()
    if lc in genus_hierarchy.index:
        return int(genus_hierarchy.loc[lc, 'genus_id']), False, genus_name
    if lc in proposed_genera:
        return proposed_genera[lc], True, genus_name
    global next_genus_id
    genus_id = next_genus_id
    next_genus_id += 1
    proposed_genera[lc] = genus_id
    new_records.append(dict(table='tbl_taxa_tree_genera', id_column='genus_id', id=genus_id,
                             name=genus_name, language_id=None, parent_table='tbl_taxa_tree_families',
                             parent_id=family_id, created_for=CURRENT_MANUAL_SPECIES))
    return genus_id, True, genus_name

def resolve_taxon(genus_id, species_epithet):
    """species_epithet=None means we only know the genus - look for/propose that genus's
    indeterminate-species placeholder (SEAD uses both 'indet.' and 'sp.' for this - check both)
    rather than a specific species. Returns (taxon_id, is_new, species_text_used)."""
    target_lc = (species_epithet or 'indet.').lower().rstrip('.')
    genus_taxa = taxa_master[taxa_master['genus_id'] == genus_id]
    exact = genus_taxa[genus_taxa['species'].str.lower().str.rstrip('.') == target_lc]
    if len(exact):
        return int(exact.iloc[0]['taxon_id']), False, exact.iloc[0]['species']
    if species_epithet is None:
        placeholder_existing = genus_taxa[
            genus_taxa['species'].str.lower().str.startswith(('indet', 'sp.', 'spp.'))
        ]
        if len(placeholder_existing):
            row0 = placeholder_existing.iloc[0]
            return int(row0['taxon_id']), False, row0['species']
    key = (genus_id, target_lc)
    species_text = species_epithet or 'indet.'
    if key in proposed_taxa:
        return proposed_taxa[key], True, species_text
    global next_taxon_id
    taxon_id = next_taxon_id
    next_taxon_id += 1
    proposed_taxa[key] = taxon_id
    new_records.append(dict(table='tbl_taxa_tree_master', id_column='taxon_id', id=taxon_id,
                             name=species_text, language_id=None, parent_table='tbl_taxa_tree_genera',
                             parent_id=genus_id, created_for=CURRENT_MANUAL_SPECIES))
    return taxon_id, True, species_text

def find_genus_default(genus_name):
    """Look up GENUS_DEFAULT_COMMON_NAME for an already-existing SEAD common name to reuse for a
    genus-only match, rather than fabricating a new placeholder taxon. Returns a row from
    common_names_all if found, else None."""
    if genus_name not in GENUS_DEFAULT_COMMON_NAME:
        return None
    target = GENUS_DEFAULT_COMMON_NAME[genus_name].lower()
    match = common_names_all[
        (common_names_all['language_id'] == 2) & (common_names_all['common_name'].str.lower() == target)
    ]
    return match.iloc[0] if len(match) else None

def resolve_common_name(taxon_id, common_name_text, language_id, taxon_is_new):
    common_name_lc = common_name_text.rstrip('?').strip().lower()
    if not taxon_is_new:
        existing = common_names_all[
            (common_names_all['taxon_id'] == taxon_id) & (common_names_all['language_id'] == language_id)
        ]
        if len(existing):
            return int(existing.iloc[0]['taxon_common_name_id']), False
    key = (taxon_id, language_id, common_name_lc)
    if key in proposed_common_names:
        return proposed_common_names[key], True
    global next_common_name_id
    common_name_id = next_common_name_id
    next_common_name_id += 1
    proposed_common_names[key] = common_name_id
    new_records.append(dict(table='tbl_taxa_common_names', id_column='taxon_common_name_id', id=common_name_id,
                             name=common_name_text, language_id=language_id,
                             parent_table='tbl_taxa_tree_master', parent_id=taxon_id,
                             created_for=CURRENT_MANUAL_SPECIES))
    return common_name_id, True


In [19]:
WEAK_GBIF_MATCH_TYPES = {'FUZZY', 'VERNACULAR'}

def get_known_names(row):
    # Fall back to GBIF's own genus/family/order whenever SEAD gave us nothing at that rank - see
    # the ANIMAL_LATIN dictionary matches (e.g. 'ko' -> Bos taurus), which only ever carry a
    # species binomial with no family/order from the dictionary itself, but step 4's GBIF
    # cross-check on that binomial already resolved the real family/order (Bovidae/Artiodactyla).
    genus = row['genus'] if pd.notna(row['genus']) else (row['gbif_genus'] if pd.notna(row['gbif_genus']) else None)
    family = row['family'] if pd.notna(row['family']) else (row['gbif_family'] if pd.notna(row['gbif_family']) else None)
    order = row['order'] if pd.notna(row['order']) else (row['gbif_order'] if pd.notna(row['gbif_order']) else None)
    species_epithet = None
    if pd.notna(row['sead_species_name']):
        parts = row['sead_species_name'].split(None, 1)
        if len(parts) == 2:
            genus = genus or parts[0]
            species_epithet = parts[1]
    return genus, family, order, species_epithet

def common_name_text_and_language(row, manual_species):
    """manual_species is only usable as a Swedish common name when it was actually matched as
    Swedish text (a real common name, a suffix-inferred/ambiguous compound, or an exact GBIF
    vernacular hit). A '(latin)' match_level means manual_species IS the raw Latin genus/family/
    order name the researcher typed (e.g. 'salix', 'salix sp', 'cf picea sp') - not a common name
    in any language - so for those, use GBIF's English vernacular name instead, if one was found."""
    match_level = row.get('match_level')
    is_latin_match = isinstance(match_level, str) and '(latin)' in match_level
    if not is_latin_match:
        return manual_species.rstrip('?').strip(), 2
    english = row.get('species_split_english')
    if pd.notna(english):
        return english, 1
    return None, None

def resolve_row(row):
    global CURRENT_MANUAL_SPECIES
    manual_species = row['manual_species']
    empty_result = dict(
        resolved_order=pd.NA, resolved_order_is_new=False,
        resolved_family=pd.NA, resolved_family_is_new=False,
        resolved_genus=pd.NA, resolved_genus_is_new=False,
        resolved_species=pd.NA, resolved_scientific_name=pd.NA,
        resolved_taxon_id=pd.NA, taxon_id_is_new=False,
        matched_existing_taxon=False,
        common_name_id=pd.NA, common_name_id_is_new=False,
        common_name_text=pd.NA, common_name_language=pd.NA,
        taxonomy_changes_needed=pd.NA,
        needs_manual_review=False, review_reason=pd.NA,
    )
    if pd.isna(manual_species):
        return pd.Series(empty_result)

    CURRENT_MANUAL_SPECIES = manual_species
    genus, family, order, species_epithet = get_known_names(row)
    genus_known, family_known, order_known = genus is not None, family is not None, order is not None

    review_reasons = []
    matched_existing_taxon = False
    order_id = order_is_new = order_name_used = None
    family_id = family_is_new = family_name_used = None
    genus_id = genus_is_new = genus_name_used = None
    species_used = None
    taxon_id, taxon_is_new = None, False

    if pd.notna(row['taxon_id']):
        # species (common name) match already gave us this straight from SEAD - nothing to do.
        taxon_id = int(row['taxon_id'])
        genus_name_used = genus
        species_used = row['sead_species_name'].split(None, 1)[1] if pd.notna(row['sead_species_name']) else None
        matched_existing_taxon = True
    elif genus_known or family_known or order_known:
        order_id, order_is_new, order_name_used = resolve_order(order)
        family_id, family_is_new, family_name_used = resolve_family(family, order_id, order)
        genus_id, genus_is_new, genus_name_used = resolve_genus(genus, family_id, family_name_used)

        default_match = find_genus_default(genus_name_used) if (species_epithet is None and genus_known) else None
        if default_match is not None:
            taxon_id = int(default_match['taxon_id'])
            species_used = taxa_master.loc[taxa_master['taxon_id'] == taxon_id, 'species'].iloc[0]
            matched_existing_taxon = True
        else:
            taxon_id, taxon_is_new, species_used = resolve_taxon(genus_id, species_epithet)

    common_name_id, common_name_is_new, common_name_text, common_name_language = None, False, None, None
    if taxon_id is not None:
        if matched_existing_taxon and not taxon_is_new:
            existing = common_names_all[
                (common_names_all['taxon_id'] == taxon_id) & (common_names_all['language_id'] == 2)
            ]
            if len(existing):
                common_name_id = int(existing.iloc[0]['taxon_common_name_id'])
                common_name_text = existing.iloc[0]['common_name']
                common_name_language = 2
        if common_name_id is None:
            text, lang = common_name_text_and_language(row, manual_species)
            if text is not None:
                common_name_id, common_name_is_new = resolve_common_name(taxon_id, text, lang, taxon_is_new)
                common_name_text, common_name_language = text, lang
            else:
                review_reasons.append('no Swedish or English common name text available to attach')

    changes = []
    if order_is_new:
        changes.append(f"new order '{order_name_used}'")
    if family_is_new:
        changes.append(f"new family '{family_name_used}'")
    if genus_is_new:
        changes.append(f"new genus '{genus_name_used}'")
    if taxon_is_new:
        changes.append(f"new taxon '{genus_name_used} {species_used}'")
    if common_name_is_new:
        lang_label = LANGUAGE_NAMES.get(common_name_language, common_name_language)
        changes.append(f"new {lang_label} common name '{common_name_text}'")
    taxonomy_changes_needed = '; '.join(changes) if changes else None

    if taxon_id is None:
        review_reasons.append('no GBIF or SEAD match at any rank')
    if row.get('match_level') == 'genus (suffix, ambiguous)':
        candidates = row.get('ambiguous_genus_candidates')
        if pd.notna(candidates):
            review_reasons.append(f'ambiguous genus suffix match, candidates: {candidates}')
        else:
            review_reasons.append('ambiguous genus suffix match (candidates not available)')
    if row.get('gbif_match_type') in WEAK_GBIF_MATCH_TYPES:
        review_reasons.append(f"weak GBIF match ({row['gbif_match_type']})")
    elif pd.notna(row.get('gbif_confidence')) and row['gbif_confidence'] < 90:
        review_reasons.append(f"low GBIF confidence ({row['gbif_confidence']})")
    if manual_species.endswith('?'):
        review_reasons.append("marked uncertain ('?') in the manual mapping")

    resolved_scientific_name = (
        f'{genus_name_used} {species_used}' if genus_name_used and species_used else pd.NA
    )

    return pd.Series(dict(
        resolved_order=order_name_used, resolved_order_is_new=bool(order_is_new),
        resolved_family=family_name_used, resolved_family_is_new=bool(family_is_new),
        resolved_genus=genus_name_used, resolved_genus_is_new=bool(genus_is_new),
        resolved_species=species_used, resolved_scientific_name=resolved_scientific_name,
        resolved_taxon_id=taxon_id, taxon_id_is_new=taxon_is_new,
        matched_existing_taxon=matched_existing_taxon,
        common_name_id=common_name_id, common_name_id_is_new=common_name_is_new,
        common_name_text=common_name_text, common_name_language=common_name_language,
        taxonomy_changes_needed=taxonomy_changes_needed,
        needs_manual_review=len(review_reasons) > 0,
        review_reason='; '.join(review_reasons) if review_reasons else pd.NA,
    ))

resolution = manual_species_taxa.apply(resolve_row, axis=1)
# manual_species_taxa already has 'taxon_id'/'genus'/'family'/'order' columns from steps 3-4;
# resolve_row's output is the final, authoritative version of all of these (now clearly labelled
# resolved_*, alongside the actual species/scientific name so the rank is easy to read), so drop
# the old ones before joining rather than ending up with duplicate columns.
manual_species_taxa = manual_species_taxa.drop(columns=['taxon_id', 'genus', 'family', 'order']).join(resolution)
manual_species_taxa = manual_species_taxa.rename(columns={'resolved_taxon_id': 'taxon_id'})

print(f"{manual_species_taxa['taxon_id'].notna().sum()} of {len(manual_species_taxa)} rows resolved to a taxon_id "
      f"({int(manual_species_taxa['taxon_id_is_new'].sum())} newly proposed, "
      f"{int(manual_species_taxa['matched_existing_taxon'].sum())} matched an existing SEAD taxon)")
print(f"{int(manual_species_taxa['needs_manual_review'].sum())} rows flagged needs_manual_review")
print(f'{len(new_records)} new SEAD records proposed across all tables')


158 of 237 rows resolved to a taxon_id (53 newly proposed, 64 matched an existing SEAD taxon)
99 rows flagged needs_manual_review
194 new SEAD records proposed across all tables


In [20]:
manual_species_taxa.to_csv(OUTPUT_DIR / 'manual_species_sead_taxa_matches.csv', index=False)

new_sead_records = pd.DataFrame(
    new_records,
    columns=['table', 'id_column', 'id', 'name', 'language_id', 'parent_table', 'parent_id', 'created_for'],
)
new_sead_records.to_csv(OUTPUT_DIR / 'new_sead_records.csv', index=False)

print(f"Saved {len(manual_species_taxa)} rows to manual_species_sead_taxa_matches.csv")
print(f"Saved {len(new_sead_records)} proposed new records to new_sead_records.csv")
new_sead_records['table'].value_counts()

Saved 237 rows to manual_species_sead_taxa_matches.csv
Saved 194 proposed new records to new_sead_records.csv


table
tbl_taxa_common_names     81
tbl_taxa_tree_master      45
tbl_taxa_tree_genera      32
tbl_taxa_tree_families    22
tbl_taxa_tree_orders      14
Name: count, dtype: int64

## Summary

In [21]:
has_species = manual_species_taxa[manual_species_taxa['manual_species'].notna()]
existing_taxon = has_species[has_species['taxon_id'].notna() & ~has_species['taxon_id_is_new']]
new_taxon = has_species[has_species['taxon_id_is_new']]
no_taxon = has_species[has_species['taxon_id'].isna()]

print(f'{len(has_species)} manual_species values (excluding the blank/no-value row)')
print(f'  {len(existing_taxon)} resolved to an existing SEAD taxon_id')
print(f'  {len(new_taxon)} resolved via a newly-proposed SEAD taxon_id')
print(f'  {len(no_taxon)} left with no taxon_id at all (no anchor found)')
print(f"  {int(has_species['needs_manual_review'].sum())} flagged needs_manual_review "
      f"(may overlap with the above - a resolved row can still be flagged, e.g. '?'-marked values)")
print()
print('needs_manual_review reasons (top patterns):')
has_species.loc[has_species['needs_manual_review'], 'review_reason'].value_counts().head(15)

236 manual_species values (excluding the blank/no-value row)
  105 resolved to an existing SEAD taxon_id
  53 resolved via a newly-proposed SEAD taxon_id
  78 left with no taxon_id at all (no anchor found)
  99 flagged needs_manual_review (may overlap with the above - a resolved row can still be flagged, e.g. '?'-marked values)

needs_manual_review reasons (top patterns):


review_reason
no GBIF or SEAD match at any rank                                                                                                                                                                                                                                                                                                                                                                                                                                                                           43
no GBIF or SEAD match at any rank; marked uncertain ('?') in the manual mapping                                                                                                                                                                                                                                                                                                                                                                                                                            